# Swahili Hate & Abuse Detection with AfroXLMR

> **Content warning:** this notebook prints abusive and hateful tweets from the dataset.

This notebook fine-tunes **AfroXLMR-base** to classify Swahili tweets from **AfriHate** as **Normal**, **Abuse** or
**Hate**. It compares the model with a TF-IDF baseline and with XLM-R, analyses its errors, and prepares the Streamlit
web app.

**How to run:** use a Colab **GPU** runtime and add two Colab secrets: `HF_TOKEN` (read) and `HF_WRITE_TOKEN` (write,
only needed to upload the model). Then run the cells from top to bottom. The model and the cleaned splits are saved to
Google Drive under `MyDrive/swahili-hate-afroxlmr`.

## 1. Load the AfriHate Swahili data

AfriHate (Muhammad et al., NAACL 2025) contains tweets in 15 African languages, annotated by native speakers. The
cells below find the Swahili files on the Hugging Face Hub, log in, and load the train, dev and test splits.

In [ ]:
from huggingface_hub import list_repo_files

files = list_repo_files("afrihate/afrihate", repo_type="dataset")
print(len(files), "files")
for f in files:
    if "swa" in f.lower():
        print(f)

61 files
data/swa/dev.tsv
data/swa/test.tsv
data/swa/train.tsv


In [ ]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get("HF_TOKEN"))

In [ ]:
import pandas as pd
from huggingface_hub import hf_hub_download

paths = {}
for split in ["train", "dev", "test"]:
    paths[split] = hf_hub_download("afrihate/afrihate", f"data/swa/{split}.tsv",
                                   repo_type="dataset")

train = pd.read_csv(paths["train"], sep="\t")
dev   = pd.read_csv(paths["dev"], sep="\t")
test  = pd.read_csv(paths["test"], sep="\t")

print(train.shape, dev.shape, test.shape)
print(list(train.columns))
print(train.head(3))
for c in train.columns:
    if train[c].nunique() < 10:
        print(f"\n{c}:")
        print(train[c].value_counts())

train.tsv:   0%|          | 0.00/1.66M [00:00<?, ?B/s]

dev.tsv:   0%|          | 0.00/348k [00:00<?, ?B/s]

test.tsv:   0%|          | 0.00/352k [00:00<?, ?B/s]

(14760, 3) (3164, 3) (3168, 3)
['id', 'tweet', 'label']
                    id                                              tweet  \
0  train_swahili_00001   kitita kile ninachopenda pussy huenda ukifany...   
1  train_swahili_00002  tunaongoja baba  raila odinga apig tarumbeta n...   
2  train_swahili_00003  mahakama kuu imemhukumu kunyongwa hadi kufa ma...   

    label  
0   Abuse  
1  Normal  
2  Normal  

label:
label
Normal    6586
Abuse     5394
Hate      2780
Name: count, dtype: int64


## 2. Data quality checks

Before training, check the data for missing values, duplicate tweets, tweets that appear with **conflicting labels**,
and tweets that appear in **more than one split**. Overlap between splits is leakage: the model would be tested on
tweets it has already seen in training, which inflates the scores.

In [ ]:
for name, df in [("train", train), ("dev", dev), ("test", test)]:
    words = df["tweet"].fillna("").str.split().str.len()
    print(f"\n=== {name}: {df.shape} ===")
    print("missing:", df.isnull().sum().to_dict())
    print("duplicate tweets:", df["tweet"].duplicated().sum())
    print("words per tweet (mean/median/max):", int(words.mean()), int(words.median()), words.max())
    print((df["label"].value_counts(normalize=True) * 100).round(1).to_dict())

# Conflicting labels for the same tweet
conf = train.groupby("tweet")["label"].nunique()
print("\nTrain tweets with conflicting labels:", (conf > 1).sum())

# Leakage between splits
tr = set(train["tweet"])
print("Train/dev overlap: ", len(tr & set(dev["tweet"])))
print("Train/test overlap:", len(tr & set(test["tweet"])))
print("Dev/test overlap:  ", len(set(dev["tweet"]) & set(test["tweet"])))


=== train: (14760, 3) ===
missing: {'id': 0, 'tweet': 0, 'label': 0}
duplicate tweets: 838
words per tweet (mean/median/max): 13 12 83
{'Normal': 44.6, 'Abuse': 36.5, 'Hate': 18.8}

=== dev: (3164, 3) ===
missing: {'id': 0, 'tweet': 0, 'label': 0}
duplicate tweets: 37
words per tweet (mean/median/max): 13 12 44
{'Normal': 44.6, 'Abuse': 36.5, 'Hate': 18.9}

=== test: (3168, 3) ===
missing: {'id': 0, 'tweet': 0, 'label': 0}
duplicate tweets: 41
words per tweet (mean/median/max): 13 12 83
{'Normal': 44.6, 'Abuse': 36.6, 'Hate': 18.8}

Train tweets with conflicting labels: 31
Train/dev overlap:  347
Train/test overlap: 387
Dev/test overlap:   71


## 3. Cleaning and leakage removal

Tweets are compared after lower-casing and collapsing spaces, so that near-identical copies match. The cleaning then:

1. drops tweets that have conflicting labels,
2. keeps one copy of each duplicate,
3. removes dev tweets that also appear in train,
4. removes test tweets that also appear in train or dev.

All models below are trained and evaluated on these cleaned splits.

In [ ]:
import re

def key(s):
    return re.sub(r"\s+", " ", str(s).lower()).strip()

def clean(df):
    df = df.copy()
    df["key"] = df["tweet"].map(key)
    conf = df.groupby("key")["label"].nunique()
    bad = set(conf[conf > 1].index)
    df = df[~df["key"].isin(bad)]
    df = df.drop_duplicates(subset="key").reset_index(drop=True)
    return df, len(bad)

before = {"train": len(train), "dev": len(dev), "test": len(test)}

train_c, b1 = clean(train)
dev_c,   b2 = clean(dev)
test_c,  b3 = clean(test)

# remove leakage: train -> dev/test, then dev -> test
tr_keys = set(train_c["key"])
dev_c  = dev_c[~dev_c["key"].isin(tr_keys)].reset_index(drop=True)
test_c = test_c[~test_c["key"].isin(tr_keys)].reset_index(drop=True)
test_c = test_c[~test_c["key"].isin(set(dev_c["key"]))].reset_index(drop=True)

print("conflicting-label tweets dropped: train", b1, "| dev", b2, "| test", b3)
print("Before:", before)
print("After: ", {"train": len(train_c), "dev": len(dev_c), "test": len(test_c)})
for name, df in [("train", train_c), ("dev", dev_c), ("test", test_c)]:
    print(name, (df["label"].value_counts(normalize=True) * 100).round(1).to_dict())

conflicting-label tweets dropped: train 31 | dev 3 | test 2
Before: {'train': 14760, 'dev': 3164, 'test': 3168}
After:  {'train': 13876, 'dev': 2766, 'test': 2664}
train {'Normal': 43.7, 'Abuse': 38.5, 'Hate': 17.8}
dev {'Normal': 42.0, 'Abuse': 41.0, 'Hate': 17.0}
test {'Abuse': 42.6, 'Normal': 41.5, 'Hate': 15.9}


## 4. Baseline: TF-IDF + logistic regression

A simple, fast baseline: TF-IDF features over word unigrams and bigrams, fed to a logistic regression with balanced
class weights, because Hate is the smallest class. It shows how much can be done with word cues alone, and the
Transformer has to beat it.

The main metric is **macro-F1**, the average F1 over the three classes. It gives the minority Hate class the same
weight as Normal, which plain accuracy does not.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score

vec = TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=2, sublinear_tf=True)
X_train = vec.fit_transform(train_c["tweet"])
X_dev   = vec.transform(dev_c["tweet"])
X_test  = vec.transform(test_c["tweet"])

clf = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42)
clf.fit(X_train, train_c["label"])

for name, X, y in [("dev", X_dev, dev_c["label"]), ("test", X_test, test_c["label"])]:
    pred = clf.predict(X)
    print(f"\n=== {name} ===")
    print("accuracy:   ", round(accuracy_score(y, pred), 4))
    print("macro F1:   ", round(f1_score(y, pred, average="macro"), 4))
    print("weighted F1:", round(f1_score(y, pred, average="weighted"), 4))

pred = clf.predict(X_test)
print("\nTest report:")
print(classification_report(test_c["label"], pred, digits=3))
print("Confusion matrix (rows=true, cols=pred; order:", list(clf.classes_), ")")
print(confusion_matrix(test_c["label"], pred, labels=clf.classes_))


=== dev ===
accuracy:    0.8821
macro F1:    0.876
weighted F1: 0.8819

=== test ===
accuracy:    0.8855
macro F1:    0.8765
weighted F1: 0.8852

Test report:
              precision    recall  f1-score   support

       Abuse      0.905     0.891     0.898      1135
        Hate      0.888     0.804     0.844       423
      Normal      0.866     0.911     0.888      1106

    accuracy                          0.886      2664
   macro avg      0.886     0.869     0.877      2664
weighted avg      0.886     0.886     0.885      2664

Confusion matrix (rows=true, cols=pred; order: ['Abuse', 'Hate', 'Normal'] )
[[1011   21  103]
 [  30  340   53]
 [  76   22 1008]]


## 5. Fine-tuning AfroXLMR-base

- **Model:** `Davlan/afro-xlmr-base` is XLM-RoBERTa-base (12 layers, hidden size 768, about 278M parameters, 250k
  SentencePiece vocabulary) with further masked-language-model pre-training on 17 African languages, including Swahili
  (Alabi et al., 2022).
- **Input: output:** a tweet is tokenized and truncated to 64 tokens (tweets average 13 words). The encoder produces
  contextual embeddings, a classification head on the `<s>` token produces 3 logits, and softmax turns them into
  probabilities for Abuse, Hate and Normal.
- **Training:** cross-entropy loss, AdamW, learning rate 2e-5, batch size 16, 4 epochs, weight decay 0.01, 350 warm-up
  steps (about 10%), fp16. Dev macro-F1 is measured after every epoch and the best checkpoint is kept.
- **Load report:** the `MISSING` lines for `classifier.*` are expected, because the classification head is new and is
  learned during fine-tuning. The `UNEXPECTED` `lm_head.*` weights belong to the pre-training head and are not used.

In [ ]:
!pip install -q transformers datasets accelerate scikit-learn

In [ ]:
import numpy as np, torch
from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, set_seed)

set_seed(42)
MODEL = "Davlan/afro-xlmr-base"
MAX_LEN = 64

labels = sorted(train_c["label"].unique())          # ['Abuse', 'Hate', 'Normal']
label2id = {l: i for i, l in enumerate(labels)}
id2label = {i: l for l, i in label2id.items()}

tok = AutoTokenizer.from_pretrained(MODEL)

def prep(df):
    d = Dataset.from_pandas(df[["tweet", "label"]].reset_index(drop=True))
    d = d.map(lambda x: {"labels": label2id[x["label"]]})
    d = d.map(lambda x: tok(x["tweet"], truncation=True, max_length=MAX_LEN), batched=True)
    return d.remove_columns(["tweet", "label"])

ds_train, ds_dev, ds_test = prep(train_c), prep(dev_c), prep(test_c)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL, num_labels=len(labels), id2label=id2label, label2id=label2id)

def metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    return {"accuracy": accuracy_score(p.label_ids, preds),
            "macro_f1": f1_score(p.label_ids, preds, average="macro")}

args = TrainingArguments(
    output_dir="afroxlmr-afrihate-swa",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    num_train_epochs=4,
    weight_decay=0.01,
    warmup_steps=350,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    save_total_limit=1,
    seed=42,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer = Trainer(model=model, args=args, train_dataset=ds_train,
                  eval_dataset=ds_dev, processing_class=tok, compute_metrics=metrics)
trainer.train()

print("DEV :", trainer.evaluate(ds_dev))
print("TEST:", trainer.evaluate(ds_test, metric_key_prefix="test"))

out = trainer.predict(ds_test)
y_true, y_pred = out.label_ids, np.argmax(out.predictions, axis=1)
print(classification_report(y_true, y_pred, target_names=labels, digits=3))
print("Confusion matrix (order:", labels, ")")
print(confusion_matrix(y_true, y_pred))

config.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/398 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

...

Training Loss,Validation Loss,Epoch,Accuracy,Macro F1
0.207345,0.380145,4,0.898409,0.893933


DEV : {'eval_loss': 0.38014474511146545, 'eval_accuracy': 0.898409255242227, 'eval_macro_f1': 0.8939331235738096}


Training Loss,Validation Loss,Epoch,Accuracy,Macro F1
0.207345,0.347684,4,0.907282,0.899157


TEST: {'test_loss': 0.347683846950531, 'test_accuracy': 0.9072822822822822, 'test_macro_f1': 0.8991574582105143}


              precision    recall  f1-score   support

       Abuse      0.899     0.937     0.918      1135
        Hate      0.899     0.842     0.869       423
      Normal      0.919     0.901     0.910      1106

    accuracy                          0.907      2664
   macro avg      0.906     0.893     0.899      2664
weighted avg      0.907     0.907     0.907      2664

Confusion matrix (order: ['Abuse', 'Hate', 'Normal'] )
[[1064   14   57]
 [  36  356   31]
 [  83   26  997]]


### Save the best model and the cleaned splits to Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
SAVE_DIR = "/content/drive/MyDrive/swahili-hate-afroxlmr"
os.makedirs(SAVE_DIR, exist_ok=True)

trainer.save_model(SAVE_DIR)      # best checkpoint (dev macro F1 0.894)
tok.save_pretrained(SAVE_DIR)

train_c[["id", "tweet", "label"]].to_csv(f"{SAVE_DIR}/train_clean.csv", index=False)
dev_c[["id", "tweet", "label"]].to_csv(f"{SAVE_DIR}/dev_clean.csv", index=False)
test_c[["id", "tweet", "label"]].to_csv(f"{SAVE_DIR}/test_clean.csv", index=False)
print("saved to", SAVE_DIR, os.listdir(SAVE_DIR))

Mounted at /content/drive


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved to /content/drive/MyDrive/swahili-hate-afroxlmr ['config.json', 'model.safetensors', 'tokenizer_config.json', 'tokenizer.json', 'training_args.bin', 'train_clean.csv', 'dev_clean.csv', 'test_clean.csv', 'test_errors.csv']


## 6. Experiments

### 6.1 How much does leakage inflate the scores?

Retrain the baseline on the **raw** train set, then score it on the raw test set, on the cleaned test set, and
separately on the leaked test tweets (those also in train) and the fresh ones.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score

# Same baseline settings, but trained on the RAW (uncleaned) train set
vec_raw = TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=2, sublinear_tf=True)
Xtr = vec_raw.fit_transform(train["tweet"])
clf_raw = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42)
clf_raw.fit(Xtr, train["label"])

def report(name, df):
    pred = clf_raw.predict(vec_raw.transform(df["tweet"]))
    print(f"{name:32s} acc={accuracy_score(df['label'], pred):.4f}  "
          f"macroF1={f1_score(df['label'], pred, average='macro'):.4f}")

report("raw-trained, RAW test", test)
report("raw-trained, CLEAN test", test_c)

# The test tweets that also appear in train (the leaked ones)
tr_keys = set(train["tweet"].map(key))
leaked = test[test["tweet"].map(key).isin(tr_keys)]
fresh  = test[~test["tweet"].map(key).isin(tr_keys)]
print("\nleaked test tweets:", len(leaked), "| fresh:", len(fresh))
report("raw-trained, LEAKED tweets only", leaked)
report("raw-trained, FRESH tweets only", fresh)

raw-trained, RAW test            acc=0.8999  macroF1=0.8959
raw-trained, CLEAN test          acc=0.8874  macroF1=0.8785

leaked test tweets: 391 | fresh: 2777
raw-trained, LEAKED tweets only  acc=0.9744  macroF1=0.9631
raw-trained, FRESH tweets only   acc=0.8894  macroF1=0.8819


**Result:** the baseline gets leaked test tweets right 97.4% of the time, against 88.9% for fresh ones. Its macro-F1
is 0.896 on the raw test set but 0.879 on the cleaned one. Leakage therefore adds about 1.7 points, which is why every
model is scored on the cleaned test set.

### 6.2 Does African-language pre-training help? XLM-R base vs. AfroXLMR-base

`run()` repeats the fine-tuning recipe for any model and seed, with warm-up set to 10% of the training steps.
Here only the pre-trained model changes: XLM-R base is AfroXLMR without the African-language adaptation.

In [ ]:
import gc

def run(model_name, lr=2e-5, epochs=4, seed=42, max_len=64, tag="run"):
    set_seed(seed)
    tk = AutoTokenizer.from_pretrained(model_name)

    def prep_(df):
        d = Dataset.from_pandas(df[["tweet", "label"]].reset_index(drop=True))
        d = d.map(lambda x: {"labels": label2id[x["label"]]})
        d = d.map(lambda x: tk(x["tweet"], truncation=True, max_length=max_len), batched=True)
        return d.remove_columns(["tweet", "label"])

    d_tr, d_dev, d_te = prep_(train_c), prep_(dev_c), prep_(test_c)
    m = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=len(labels), id2label=id2label, label2id=label2id)

    steps = int(0.1 * (len(train_c) / 16) * epochs)
    a = TrainingArguments(
        output_dir=f"out-{tag}", learning_rate=lr, num_train_epochs=epochs,
        per_device_train_batch_size=16, per_device_eval_batch_size=64,
        weight_decay=0.01, warmup_steps=steps, eval_strategy="epoch",
        save_strategy="epoch", load_best_model_at_end=True,
        metric_for_best_model="macro_f1", save_total_limit=1, seed=seed,
        fp16=torch.cuda.is_available(), report_to="none")

    t = Trainer(model=m, args=a, train_dataset=d_tr, eval_dataset=d_dev,
                processing_class=tk, compute_metrics=metrics)
    t.train()
    dv = t.evaluate(d_dev)
    ts = t.evaluate(d_te, metric_key_prefix="test")
    res = {"tag": tag, "dev_f1": round(dv["eval_macro_f1"], 4),
           "test_f1": round(ts["test_macro_f1"], 4),
           "test_acc": round(ts["test_accuracy"], 4)}
    print("\nRESULT:", res)
    del t, m
    gc.collect()
    torch.cuda.empty_cache()
    return res

res_xlmr = run("xlm-roberta-base", tag="xlmr-base")

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

Map:   0%|          | 0/13876 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Accuracy,Macro F1
0.250710,0.361952,4,0.901302,0.896108


Training Loss,Validation Loss,Epoch,Accuracy,Macro F1
0.250710,0.346095,4,0.902402,0.892164



RESULT: {'tag': 'xlmr-base', 'dev_f1': 0.8961, 'test_f1': 0.8922, 'test_acc': 0.9024}


### 6.3 How much do results change between random seeds?

Re-run AfroXLMR-base with seed 123. If the gap between seeds is as large as the gap between models, the model
comparison above is not conclusive.

In [ ]:
res_afro_s2 = run("Davlan/afro-xlmr-base", seed=123, tag="afro-seed123")

Map:   0%|          | 0/13876 [00:00<?, ? examples/s]

Map:   0%|          | 0/13876 [00:00<?, ? examples/s]

Map:   0%|          | 0/2766 [00:00<?, ? examples/s]

Map:   0%|          | 0/2766 [00:00<?, ? examples/s]

Map:   0%|          | 0/2664 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Accuracy,Macro F1
0.211063,0.383231,4,0.899855,0.895154


Training Loss,Validation Loss,Epoch,Accuracy,Macro F1
0.211063,0.359431,4,0.901652,0.891728



RESULT: {'tag': 'afro-seed123', 'dev_f1': 0.8952, 'test_f1': 0.8917, 'test_acc': 0.9017}


### Experiment summary

| Model | Seed | Dev macro-F1 | Test accuracy | Test macro-F1 |
|---|---|---|---|---|
| TF-IDF + logistic regression (baseline) | – | 0.8760 | 0.8855 | 0.8765 |
| AfroXLMR-base (**deployed**) | 42 | 0.8939 | 0.9073 | 0.8992 |
| XLM-R base | 42 | 0.8961 | 0.9024 | 0.8922 |
| AfroXLMR-base | 123 | 0.8952 | 0.9017 | 0.8917 |

- The Transformers beat the baseline by 1.5–2.3 macro-F1 points, mostly through better Hate and Abuse recall.
- AfroXLMR and XLM-R cannot be told apart: changing the seed moves AfroXLMR by 0.0075, about as much as the gap between
  the two models (0.0070), and XLM-R is slightly ahead on dev. Swahili was already in XLM-R's pre-training data, which
  may explain the small difference.
- The deployed run (seed 42) is the best of three on test, so the AfroXLMR mean (0.8955) is the fairer estimate.

## 7. Error analysis

Run the saved model over the cleaned test set, save the misclassified tweets, and read samples of each error type.
Missed hate (true Hate, predicted Normal) is the costliest error for a moderation tool.

In [ ]:
import textwrap
from transformers import pipeline

SAVE_DIR = "/content/drive/MyDrive/swahili-hate-afroxlmr"
pipe = pipeline("text-classification", model=SAVE_DIR, tokenizer=SAVE_DIR,
                truncation=True, max_length=64, batch_size=64,
                device=0 if torch.cuda.is_available() else -1)

preds = pipe(test_c["tweet"].tolist())
test_c["pred"] = [p["label"] for p in preds]
test_c["conf"] = [round(p["score"], 3) for p in preds]

wrong = test_c[test_c["label"] != test_c["pred"]].copy()
wrong.to_csv(f"{SAVE_DIR}/test_errors.csv", index=False)
print("errors:", len(wrong), "of", len(test_c))
print(wrong.groupby(["label", "pred"]).size().sort_values(ascending=False))

def show(true, pred, n=8):
    sub = wrong[(wrong["label"] == true) & (wrong["pred"] == pred)]
    print(f"\n{'='*60}\nTRUE: {true} -> PRED: {pred}  ({len(sub)} cases)\n{'='*60}")
    for _, r in sub.sample(min(n, len(sub)), random_state=42).iterrows():
        print(f"[conf {r['conf']}] " + textwrap.fill(r["tweet"], 100))
        print("-" * 40)

show("Hate", "Normal")      # missed hate: the costliest error
show("Normal", "Abuse")     # false alarms

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

errors: 248 of 2664
label   pred  
Normal  Abuse     83
Abuse   Normal    58
Hate    Abuse     36
        Normal    31
Normal  Hate      26
Abuse   Hate      14
dtype: int64

TRUE: Hate -> PRED: Normal  (31 cases)
[conf 0.878] wembe ya wizi uwa yaisha makali kwa haraka now we know why wembe ni wewe na sio ule ule
electionsboycottke
----------------------------------------
[conf 0.893] when i saw kalonzo holding a placard nlikua nmeona ka ni wale wasee huajiriwa kukaribisha customers
kwa pizza shops nasasinkingship
----------------------------------------
[conf 0.97] Ni kama wanawake hawakuumbwa na mungu
----------------------------------------
[conf 0.838] wewe jembe jinga  ikiwa unachukua teksi kwenda kwa kilabu
----------------------------------------
[conf 0.987] wakenya waliitikia wito wetu wa kutopiga kura na kukaa nyumbani  raila odinga lazima abaki
----------------------------------------
[conf 0.996]  ughna lemme nadhani wana keki ya manjano pia sikia hadithi hii hapo awali
---

In [ ]:
show("Abuse", "Normal")
show("Hate", "Abuse")

print("\nMean confidence, correct predictions:", test_c[test_c["label"] == test_c["pred"]]["conf"].mean().round(3))
print("Mean confidence, wrong predictions:  ", wrong["conf"].mean().round(3))
print("Wrong predictions with conf > 0.9:   ", (wrong["conf"] > 0.9).mean().round(3))


TRUE: Abuse -> PRED: Normal  (58 cases)
[conf 0.778] kama negro  moja ya vitu vya kwanza wanavyoangalia ni viatu mbwa
----------------------------------------
[conf 0.547] mtu mwambie huyu bwege kwa stfu
----------------------------------------
[conf 0.503] wasichana hawa wadogo huko simons walikuwa wakiniambia wanapenda soksi zangu za huf bitch nenda
nyumbani  amp fanya kazi yako ya nyumbani
----------------------------------------
[conf 0.707] wakati nigga yako inakupitisha jembe  jinga hilo
----------------------------------------
[conf 0.984] yeye ni mtoto mdogo amechoka tu na ujinga wako mtazamo wake kwako unategemea jinsi unavyomtendea
----------------------------------------
[conf 0.768]  blackmantide ha bitch mimi nina watoto wachanga wa pwani kushoto n kulia kwenye pwani ya magharibi
----------------------------------------
[conf 0.54] je mnapenda kusema pesa hazinunulii furaha lakini jina niggah ameketi lambo akilia
----------------------------------------
[conf 0.961] mbele

### What the errors show

- **Possible label noise:** some tweets labelled Normal contain explicit slurs or swearing, and the model calls them
  Abuse. Swahili annotator agreement in AfriHate is only moderate (kappa 0.55).
- **Implicit hate is missed:** *"Ni kama wanawake hawakuumbwa na mungu"* ("It's as if women were not created by God")
  is Hate, but it is predicted Normal with 0.97 confidence, because there is no offensive word to latch onto.
- **Hate vs. Abuse:** tweets with both profanity and an identity target tend to be predicted as Abuse.
- **Code-switching and noisy text:** English slang, usernames and hashtags inside Swahili sentences are common among
  the errors, and some tweets read like word-for-word translations from English.
- **Confidence:** wrong predictions have lower confidence on average, but many errors are still made with high confidence.

### Confidence and a human-review threshold

Is the model's confidence useful? Grouping the test tweets by confidence helps check the accuracy in each group, and see how
many errors a review threshold would catch.

In [ ]:
import pandas as pd

test_c["correct"] = test_c["label"] == test_c["pred"]
bins = [0, 0.6, 0.8, 0.9, 0.95, 0.99, 1.0]
test_c["conf_bin"] = pd.cut(test_c["conf"], bins=bins, include_lowest=True)

summary = test_c.groupby("conf_bin", observed=True).agg(
    tweets=("correct", "size"),
    accuracy=("correct", "mean"),
    errors=("correct", lambda s: (~s).sum()),
).round(3)
print(summary)

# Review threshold: what share of errors falls below each confidence cut-off?
total_err = (~test_c["correct"]).sum()
for t in [0.8, 0.9, 0.95]:
    low = test_c[test_c["conf"] < t]
    print(f"conf < {t}: {len(low)/len(test_c):.1%} of tweets flagged, "
          f"catches {(~low['correct']).sum()/total_err:.1%} of errors")

               tweets  accuracy  errors
conf_bin                               
(-0.001, 0.6]      37     0.514      18
(0.6, 0.8]         93     0.559      41
(0.8, 0.9]         83     0.578      35
(0.9, 0.95]        96     0.594      39
(0.95, 0.99]      441     0.855      64
(0.99, 1.0]      1914     0.973      51
conf < 0.8: 4.9% of tweets flagged, catches 23.8% of errors
conf < 0.9: 8.0% of tweets flagged, catches 37.5% of errors
conf < 0.95: 11.4% of tweets flagged, catches 52.8% of errors


**Result:** tweets predicted with more than 99% confidence are 97.3% correct. Sending posts below **0.95** confidence to
a human flags 11.4% of posts and catches 52.8% of errors, so the web app uses this threshold. However, 62% of the errors
(154 of 248) still have more than 90% confidence, so human review of flagged posts is still needed.

## 8. Interactive demo and model upload

A quick Gradio demo inside Colab. Its share link is temporary: it expires within a week and only works while this
runtime is running. It is not the deployment; see section 9 for that.

In [ ]:
!pip install -q gradio

In [ ]:
import gradio as gr
import torch
from transformers import pipeline

SAVE_DIR = "/content/drive/MyDrive/swahili-hate-afroxlmr"
REVIEW_THRESHOLD = 0.95

clf_pipe = pipeline("text-classification", model=SAVE_DIR, tokenizer=SAVE_DIR,
                    top_k=None, truncation=True, max_length=64,
                    device=0 if torch.cuda.is_available() else -1)

def classify(text):
    if not text or not text.strip():
        return {}, "Please enter some Swahili text."
    scores = clf_pipe(text)[0]
    probs = {s["label"]: float(s["score"]) for s in scores}
    top, conf = max(probs.items(), key=lambda kv: kv[1])
    if conf < REVIEW_THRESHOLD:
        note = f"Low confidence ({conf:.0%}): send to a human moderator for review."
    else:
        note = f"Confidence {conf:.0%}. A human should still review any flagged post."
    return probs, note

demo = gr.Interface(
    fn=classify,
    inputs=gr.Textbox(lines=4, label="Swahili post",
                      placeholder="Write a post..."),
    outputs=[gr.Label(num_top_classes=3, label="Prediction (Normal / Abuse / Hate)"),
             gr.Textbox(label="Moderation note")],
    title="Swahili Hate and Abuse Detector",
    description=("Content warning: this tool deals with hateful and abusive language. "
                 "Fine-tuned AfroXLMR on AfriHate (Swahili). It is an assistive tool for "
                 "human moderators and must not be used for automated removal."),
    examples=[["Habari za asubuhi, natumai una siku njema."]],
)

demo.launch(share=True)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9c40e1e2bd1d293869.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


### Upload the model to the Hugging Face Hub

The deployed app downloads the model from here
([ksheilla/swahili-hate-afroxlmr](https://huggingface.co/ksheilla/swahili-hate-afroxlmr)). Only the model and
tokenizer files are uploaded, not the dataset copies.

In [ ]:
from google.colab import userdata
from huggingface_hub import HfApi

api = HfApi(token=userdata.get("HF_WRITE_TOKEN"))
user = api.whoami()["name"]
repo_id = f"{user}/swahili-hate-afroxlmr"

api.create_repo(repo_id, repo_type="model", exist_ok=True)
api.upload_folder(
    folder_path="/content/drive/MyDrive/swahili-hate-afroxlmr",
    repo_id=repo_id,
    repo_type="model",
    ignore_patterns=["*.csv", "training_args.bin"],   # model files only, no dataset copies
)
print("Uploaded:", f"https://huggingface.co/{repo_id}")

No files have been modified since last commit. Skipping to prevent empty commit.


Uploaded: https://huggingface.co/ksheilla/swahili-hate-afroxlmr


## Deployment: Streamlit Community Cloud

The Hugging Face Space cell failed with `402 Payment Required`: Hugging Face now requires a paid PRO plan to host Gradio or Docker Spaces. Free accounts can only host Gradio apps on ZeroGPU, and only once the account is 30 days old. The app is therefore deployed on **Streamlit Community Cloud**, which is free and allows up to about 2.7 GB of RAM. The memory test below shows the app needs about 1.7 GB.



In [ ]:
# Replaces the Hugging Face Space cell (402 Payment Required, see the note above).
import os

DEPLOY_DIR = "/content/drive/MyDrive/swahili-hate-afroxlmr/deploy"
os.makedirs(DEPLOY_DIR, exist_ok=True)

# transformers 5.18.0 is the version the model was saved with (see its config.json)
with open(f"{DEPLOY_DIR}/requirements.txt", "w") as f:
    f.write("streamlit==1.65.0\ntransformers==5.18.0\ntorch==2.14.1\n")
print(os.listdir(DEPLOY_DIR))


['requirements.txt']


In [ ]:
%%writefile /content/drive/MyDrive/swahili-hate-afroxlmr/deploy/app.py
"""Swahili Hate & Abuse Detector (Streamlit).

Loads the fine-tuned AfroXLMR-base classifier from the Hugging Face Hub
(ksheilla/swahili-hate-afroxlmr) and labels a Swahili post as
Normal, Abuse or Hate.

Run locally:  streamlit run app.py
"""
import os

import streamlit as st
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

MODEL_ID = os.environ.get("MODEL_ID", "ksheilla/swahili-hate-afroxlmr")
MAX_LEN = 64              # same max_length used during fine-tuning
REVIEW_THRESHOLD = 0.95   # from the confidence analysis: below this, a human should review

LABEL_HELP = {
    "Normal": "No abusive or hateful content detected.",
    "Abuse": "Offensive or insulting language that does not target a group identity.",
    "Hate": "Attacks a person or group because of an identity such as ethnicity, religion or gender.",
}
LABEL_COLOR = {"Normal": "green", "Abuse": "orange", "Hate": "red"}

# (button label, tooltip, text)
EXAMPLES = [
    ("Greeting", "Written for the demo", "Habari za asubuhi, natumai una siku njema."),
    ("Test set: Abuse", "AfriHate test set, true label: Abuse", "mtu mwambie huyu bwege kwa stfu"),
    ("Test set: Hate", "AfriHate test set, true label: Hate", "Ni kama wanawake hawakuumbwa na mungu"),
]

st.set_page_config(page_title="Swahili Hate & Abuse Detector", page_icon="🛡️")


@st.cache_resource(show_spinner="Loading the model (the first load takes about a minute)...")
def load_model():
    """Download the tokenizer and fine-tuned model once and keep them in memory."""
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID)
    model.eval()
    return tokenizer, model


def classify(text, tokenizer, model):
    """Return {label: probability} for one post."""
    inputs = tokenizer(text, truncation=True, max_length=MAX_LEN, return_tensors="pt")
    with torch.inference_mode():
        logits = model(**inputs).logits          # shape (1, 3): one score per class
    probs = torch.softmax(logits, dim=-1)[0]     # scores -> probabilities that sum to 1
    return {model.config.id2label[i]: float(p) for i, p in enumerate(probs)}


def use_example(text):
    st.session_state.text = text


st.title("Swahili Hate & Abuse Detector")
st.write("Paste a Swahili social media post to classify it as **Normal**, **Abuse** or **Hate**.")
st.caption(
    "Content warning: this tool deals with hateful and abusive language. "
    "It is an assistive tool for human moderators and must not be used for automated removal."
)

tokenizer, model = load_model()

st.write("Try an example:")
for col, (label, tip, text) in zip(st.columns(len(EXAMPLES)), EXAMPLES):
    col.button(label, help=tip, on_click=use_example, args=(text,), width="stretch")

text = st.text_area("Swahili post", key="text", height=120, placeholder="Andika chapisho hapa...")

if st.button("Classify", type="primary", key="classify"):
    if not text.strip():
        st.warning("Please enter some Swahili text.")
    else:
        probs = classify(text, tokenizer, model)
        label, conf = max(probs.items(), key=lambda kv: kv[1])

        st.subheader(f"Prediction: :{LABEL_COLOR[label]}[{label}]")
        st.caption(LABEL_HELP[label])
        for name, p in sorted(probs.items(), key=lambda kv: -kv[1]):
            st.progress(p, text=f"{name}: {p:.1%}")

        if conf < REVIEW_THRESHOLD:
            st.warning(f"Low confidence ({conf:.0%}): send to a human moderator for review.")
        else:
            st.info(f"Confidence {conf:.0%}. A human should still review any flagged post.")

with st.expander("How it works"):
    st.markdown(
        f"""
- **Model:** [AfroXLMR-base](https://huggingface.co/Davlan/afro-xlmr-base) (XLM-RoBERTa adapted to
  African languages), fine-tuned for 3-class classification on the Swahili split of
  [AfriHate](https://huggingface.co/datasets/afrihate/afrihate). Weights:
  [{MODEL_ID}](https://huggingface.co/{MODEL_ID}).
- **Pipeline:** text → SentencePiece tokenizer (max {MAX_LEN} tokens) → 12-layer Transformer encoder →
  classification head on the `<s>` token → softmax over Abuse / Hate / Normal.
- **Data:** 13,876 training tweets after removing duplicates, conflicting labels and train/test overlap.
- **Test results:** accuracy 0.907, macro-F1 0.899. TF-IDF + logistic regression baseline: 0.886 / 0.877.
- **Review threshold ({REVIEW_THRESHOLD:.0%}):** on the test set, posts below this confidence are 11.4% of
  all posts but contain 52.8% of the model's errors.
"""
    )


Overwriting /content/drive/MyDrive/swahili-hate-afroxlmr/deploy/app.py


### Memory check

Streamlit Community Cloud gives apps about 2.7 GB of RAM. These scripts load the model in a separate process and
report its real peak memory in float32 and bfloat16. The peak is read from `VmHWM`. `resource.ru_maxrss` cannot be
used here, because a script started with `!python` inherits the notebook kernel's peak; that is why it always printed
4.16 GB. In a test with a model of the same size, float32 peaked at about 1.6 GB and bfloat16 at about 2.9 GB. bf16 is
higher because the float32 file is converted while loading, so the app uses float32.

In [ ]:
script = '''
import sys, torch

def peak_gb():
    # VmHWM = the peak RAM of THIS process. resource.ru_maxrss is not reliable here: a script started
    # with `!python` inherits the notebook kernel's peak, which is why it always printed 4.16.
    for line in open("/proc/self/status"):
        if line.startswith("VmHWM"):
            return round(int(line.split()[1]) * 1024 / 1e9, 2)

from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

dtype = torch.bfloat16 if sys.argv[1] == "bf16" else torch.float32
MODEL = "ksheilla/swahili-hate-afroxlmr"
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL, dtype=dtype, low_cpu_mem_usage=True).eval()
p = pipeline("text-classification", model=model, tokenizer=tok, top_k=None,
             truncation=True, max_length=64, device=-1)
print(sys.argv[1], p("Habari za asubuhi, natumai una siku njema.")[0][:2])
print(sys.argv[1], "peak RAM (GB):", peak_gb())
'''
open("mem_test.py", "w").write(script)

983

In [ ]:
!python mem_test.py fp32
!python mem_test.py bf16

config.json: 100% 954/954 [00:00<00:00, 3.25MB/s]
tokenizer_config.json: 100% 343/343 [00:00<00:00, 1.18MB/s]

tokenizer.json: downloading bytes:   0% 0.00/17.1M [00:00<?, ?B/s]
tokenizer.json: downloading bytes: 100% 5.88M/5.88M [00:01<00:00, 4.00MB/s,  542kB/s  ]
tokenizer.json: reconstructing file: 100% 17.1M/17.1M [00:01<00:00, 11.6MB/s, 1.58MB/s  ]

model.safetensors: downloading bytes:   0% 0.00/1.11G [00:00<?, ?B/s]
model.safetensors: downloading bytes:  31% 342M/1.11G [00:02<00:03, 196MB/s, 28.6MB/s  ]
model.safetensors: downloading bytes:  33% 366M/1.11G [00:02<00:03, 202MB/s, 30.6MB/s  ]
model.safetensors: downloading bytes:  86% 955M/1.11G [00:04<00:00, 169MB/s, 74.7MB/s  ]
model.safetensors: downloading bytes:  93% 1.04G/1.11G [00:04<00:00, 196MB/s, 76.5MB/s  ]
model.safetensors: reconstructing file:  72% 805M/1.11G [00:12<00:05, 57.2MB/s, 32.5MB/s  ]
model.safetensors: downloading bytes: 100% 1.05G/1.05G [00:19<00:00, 55.1MB/s, 80.2MB/s  ]
model.safetensors: reconstructing

In [ ]:
script = '''
import sys, os, psutil, torch

def peak_gb():
    # VmHWM = the peak RAM of THIS process (ru_maxrss inherits the notebook kernel's peak).
    for line in open("/proc/self/status"):
        if line.startswith("VmHWM"):
            return round(int(line.split()[1]) * 1024 / 1e9, 2)

def rss(): return round(psutil.Process(os.getpid()).memory_info().rss / 1e9, 2)
print(sys.argv[1], "| after import torch:", rss())
from transformers import AutoTokenizer, AutoModelForSequenceClassification
print(sys.argv[1], "| after import transformers:", rss())

dtype = torch.bfloat16 if sys.argv[1] == "bf16" else torch.float32
MODEL = "ksheilla/swahili-hate-afroxlmr"
tok = AutoTokenizer.from_pretrained(MODEL)
print(sys.argv[1], "| after tokenizer:", rss())
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL, dtype=dtype, low_cpu_mem_usage=True).eval()
print(sys.argv[1], "| after model:", rss())
print(sys.argv[1], "| weights (GB):",
      round(sum(p.numel() * p.element_size() for p in model.parameters()) / 1e9, 2))
with torch.no_grad():
    model(**tok("Habari za asubuhi", return_tensors="pt", truncation=True, max_length=64))
print(sys.argv[1], "| after prediction:", rss())
print(sys.argv[1], "| peak:", peak_gb())
'''
open("mem_test2.py", "w").write(script)

1232

In [ ]:
!python mem_test2.py fp32
!python mem_test2.py bf16

fp32 | after import torch: 0.51
fp32 | after import transformers: 0.8
fp32 | after tokenizer: 1.18
Loading weights: 100% 201/201 [00:00<00:00, 12678.47it/s]
fp32 | after model: 1.2
fp32 | weights (GB): 1.11
fp32 | after prediction: 1.55
fp32 | peak: 1.55
bf16 | after import torch: 0.51
bf16 | after import transformers: 0.8
bf16 | after tokenizer: 1.18
Loading weights: 100% 201/201 [00:00<00:00, 264.67it/s]
bf16 | after model: 1.75
bf16 | weights (GB): 0.56
bf16 | after prediction: 1.76
bf16 | peak: 2.86


In [ ]:
# Optional: load the fine-tuned model in bfloat16. Not needed for deployment: the normal
# float32 model fits in Streamlit Community Cloud's memory, and converting to bfloat16 while
# loading actually RAISES peak RAM (see the bf16 "peak" lines in the memory test above).
import torch
from transformers import AutoModelForSequenceClassification

model_bf16 = AutoModelForSequenceClassification.from_pretrained(
    "ksheilla/swahili-hate-afroxlmr",   # your model on the Hub (was the placeholder "your-model-name")
    dtype=torch.bfloat16,               # `torch_dtype` is deprecated in transformers 5
)
weights_gb = sum(p.numel() * p.element_size() for p in model_bf16.parameters()) / 1e9
print(model_bf16.dtype, "| weights:", round(weights_gb, 2), "GB")


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

torch.bfloat16 | weights: 0.56 GB


### Smoke test of the web app

Run `app.py` with Streamlit's test runner and the real model from the Hub before pushing it to GitHub. The greeting
should come out Normal, while the two test-set posts are misclassified, just as in the error analysis.

In [ ]:
# Smoke test:I ran the deployment app.py here, with the real model from the Hub, before pushing it.
!pip install -q streamlit==1.65.0
from streamlit.testing.v1 import AppTest

at = AppTest.from_file(f"{DEPLOY_DIR}/app.py", default_timeout=600)
at.run()
assert not at.exception, at.exception

for text in ["Habari za asubuhi, natumai una siku njema.",
             "mtu mwambie huyu bwege kwa stfu",           # test set, true label: Abuse
             "Ni kama wanawake hawakuumbwa na mungu"]:    # test set, true label: Hate
    at.text_area(key="text").input(text).run()
    at.button(key="classify").click().run()
    assert not at.exception, at.exception
    probs = [p.proto.text for p in at.get("progress")]
    note = [w.value for w in at.warning] + [i.value for i in at.info]
    print(text, "\n   ", probs, note)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 100.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 97.8 MB/s eta 0:00:00


2026-10-06 01:54:37.433 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Habari za asubuhi, natumai una siku njema. 
    ['Normal: 99.8%', 'Abuse: 0.1%', 'Hate: 0.1%'] ['Confidence 100%. A human should still review any flagged post.']
mtu mwambie huyu bwege kwa stfu 
    ['Normal: 54.7%', 'Abuse: 43.4%', 'Hate: 1.9%'] ['Low confidence (55%): send to a human moderator for review.']
Ni kama wanawake hawakuumbwa na mungu 
    ['Normal: 97.0%', 'Hate: 2.7%', 'Abuse: 0.3%'] ['Confidence 97%. A human should still review any flagged post.']
